In [0]:
# Create schemas 'silver' and 'gold' under the 'sales' catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS sales.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS sales.gold")

# List schemas under the 'sales' catalog to verify
display(spark.sql("SHOW SCHEMAS IN sales"))

In [0]:
import re
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

MAPPING_PATH = "/Volumes/sales/mapping/master_file/master_mapping_bronze_silver_gold_olist.xlsx"

# ─────────────────────────────────────────────────────────────────────────────
# Load Bronze → Silver mapping from Excel
# ─────────────────────────────────────────────────────────────────────────────
bts = spark.sql(f"""
    SELECT
        TRIM(_c1) AS source_table,
        TRIM(_c2) AS source_col,
        TRIM(_c3) AS target_table,
        TRIM(_c4) AS target_col,
        TRIM(_c5) AS target_type,
        TRIM(_c6) AS transformation,
        TRIM(_c7) AS dq_rule
    FROM read_files(
        '{MAPPING_PATH}',
        format      => 'excel',
        header      => true,
        dataAddress => 'Bronze_to_Silver'
    )
    WHERE _c0 = 'Bronze to Silver'
      AND upper(trim(_c9)) = 'MAPPED'
      AND _c3 IS NOT NULL
      AND _c4 IS NOT NULL
""").toPandas()

# ─────────────────────────────────────────────────────────────────────────────
# Load Silver → Gold mapping from Excel
# ─────────────────────────────────────────────────────────────────────────────
stg = spark.sql(f"""
    SELECT
        TRIM(_c1) AS source_tables,
        TRIM(_c2) AS source_cols,
        TRIM(_c3) AS target_table,
        TRIM(_c4) AS target_col,
        TRIM(_c5) AS target_type,
        TRIM(_c6) AS transformation,
        TRIM(_c7) AS grain
    FROM read_files(
        '{MAPPING_PATH}',
        format      => 'excel',
        header      => true,
        dataAddress => 'Silver_to_Gold'
    )
    WHERE _c0 = 'Silver to Gold'
      AND upper(trim(_c9)) = 'MAPPED'
      AND _c3 IS NOT NULL
      AND _c4 IS NOT NULL
""").toPandas()

print(f"Bronze→Silver : {len(bts):>3} column mappings | {bts['target_table'].nunique()} tables")
print(f"Silver→Gold   : {len(stg):>3} column mappings | {stg['target_table'].nunique()} tables")
print("\nBTS tables:", sorted(bts['target_table'].unique()))
print("STG tables:", sorted(stg['target_table'].unique()))

# ─────────────────────────────────────────────────────────────────────────────
# Core engine: convert one mapping row → SQL column expression
# ─────────────────────────────────────────────────────────────────────────────
def to_sql_expr(row):
    """
    Returns a SQL expression string  "expr AS `target_col`"  or None (skip).
    Resolution order:
      1. current_timestamp audit cols
      2. Skip window / aggregation markers (table-level override handles these)
      3. Callable SQL functions  → use logic verbatim with F.expr()
      4. Pass-through / retain   → col as-is (with optional CAST)
      5. 'CAST TO <type>' pattern
      6. Source col is an arithmetic expression (derived col)
      7. Full CASE WHEN written out in logic text
      8. Type-cast fallback
      9. Default: retain as-is
    """
    src   = str(row.get("source_col",     "") or "").strip()
    tgt   = str(row.get("target_col",     "") or "").strip()
    dtype = str(row.get("target_type",    "") or "").strip().upper()
    lgc   = str(row.get("transformation", "") or "").strip()
    lgc_u = lgc.upper()

    if not tgt:
        return None

    # 1. Audit timestamp
    if "CURRENT_TIMESTAMP" in lgc_u:
        return f"CURRENT_TIMESTAMP() AS `{tgt}`"

    # 2. Skip markers → handled by table-level overrides
    if any(kw in lgc_u for kw in ["ROW_NUMBER", "WINDOW FUNCTION",
                                   "GROUP BY", "GROUP_BY", "AGGREGATION"]):
        return None

    # 2a. "WHEN x THEN y" without wrapping CASE → complete it
    if lgc_u.lstrip().startswith("WHEN "):
        return f"(CASE {lgc} ELSE NULL END) AS `{tgt}`"

    # 2b. BOOLEAN target columns: transformation IS the full condition expression
    #     (e.g. "price <= 0", "freight_value < 0") — use verbatim, not CAST
    PASS_KW_BOOL = [
        "PASS-THROUGH", "PASS THROUGH", "INHERITED", "RETAIN",
        "NO EXPLICIT", "KEEP ORIGINAL", "DIRECT", "AS-IS",
    ]
    if dtype == "BOOLEAN" and lgc and not any(kw in lgc_u for kw in PASS_KW_BOOL):
        return f"({lgc}) AS `{tgt}`"

    # 3. Direct SQL function calls — use logic verbatim
    SQL_FN = [
        "LOWER(", "UPPER(", "TRIM(", "REGEXP_REPLACE(", "COALESCE(",
        "ROUND(", "TO_DATE(", "TO_TIMESTAMP(", "HOUR(", "DATE_FORMAT(",
        "DATEDIFF(", "DAYOFWEEK(", "SUBSTRING(", "INITCAP(",
        "YEAR(", "MONTH(", "DAYOFMONTH(", "CAST(", "TRY_CAST(",
        "WEEKOFYEAR(", "QUARTER(", "NVL(", "IF(", "CONCAT(",
        "DATE_TRUNC(", "DATE_DIFF(", "CASE WHEN",
    ]
    if any(fn in lgc_u for fn in SQL_FN):
        return f"({lgc}) AS `{tgt}`"

    # 4. Pass-through variants
    PASS_KW = [
        "PASS-THROUGH", "PASS THROUGH", "INHERITED", "RETAIN",
        "NO EXPLICIT", "KEEP ORIGINAL", "DIRECT", "SELECT DISTINCT",
        "KEPT AS-IS", "KEPT AS IS", "AS-IS",
    ]
    is_passthrough = not lgc or any(kw in lgc_u for kw in PASS_KW)
    if is_passthrough:
        cast = (
            f"CAST(`{src}` AS {dtype})"
            if dtype and dtype not in ("STRING", "VARCHAR", "")
            else f"`{src}`"
        )
        return f"{cast} AS `{tgt}`"

    # 5. "CAST TO <type>" or "CAST AS <type>" pattern
    if re.match(r"^CAST\s+(TO|AS)\s+\w", lgc_u):
        return f"CAST(`{src}` AS {dtype}) AS `{tgt}`"

    # 6. Source col is an arithmetic expression (derived col: volume, etc.)
    if src and any(op in src for op in ["*", "+", "-", "/", "("]):
        if "ROUND" in lgc_u:
            m = re.search(r"ROUND.*?,\s*(\d+)", lgc_u)
            prec = m.group(1) if m else "2"
            return f"ROUND({src}, {prec}) AS `{tgt}`"
        return f"({src}) AS `{tgt}`"

    # 7. Full CASE WHEN written out in the logic text
    if "CASE" in lgc_u and "WHEN" in lgc_u and "THEN" in lgc_u:
        return f"({lgc}) AS `{tgt}`"

    # 8. Type-cast fallback
    if dtype and dtype not in ("STRING", "VARCHAR", ""):
        return f"CAST(`{src}` AS {dtype}) AS `{tgt}`"

    # 9. Default: retain as-is
    return f"`{src}` AS `{tgt}`"

print("\n✓ Mapping loaded and engine function defined")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Table-level overrides for operations that cannot be expressed as a single
# column expression (dedup windows, multi-step imputation, WHERE filters,
# centroid aggregations).  Everything else is driven purely by the Excel mapping.
# ══════════════════════════════════════════════════════════════════════════════

BTS_OVERRIDES = {

    # Sellers: remove duplicate seller_id rows
    "sales.silver.sellers": {
        "dedup_partition": "seller_id",
        "dedup_order_by":  "seller_id",
    },

    # Products: category-average null imputation + volume derivation
    "sales.silver.products": {
        "full_sql": """
            WITH base AS (
                SELECT
                    product_id,
                    COALESCE(product_category_name, 'unknown_category')         AS product_category_name,
                    TRY_CAST(product_name_lenght        AS INT)                  AS product_name_length,
                    TRY_CAST(product_description_lenght AS INT)                  AS product_description_length,
                    TRY_CAST(product_photos_qty         AS INT)                  AS product_photos_qty,
                    TRY_CAST(product_weight_g           AS DOUBLE)               AS product_weight_g,
                    TRY_CAST(product_length_cm          AS DOUBLE)               AS product_length_cm,
                    TRY_CAST(product_height_cm          AS DOUBLE)               AS product_height_cm,
                    TRY_CAST(product_width_cm           AS DOUBLE)               AS product_width_cm
                FROM sales.bronze.products
            ),
            cat_avgs AS (
                SELECT product_category_name,
                       ROUND(AVG(product_name_length),        0) AS avg_name_len,
                       ROUND(AVG(product_description_length), 0) AS avg_desc_len,
                       ROUND(AVG(product_photos_qty),         0) AS avg_photos,
                       ROUND(AVG(product_weight_g),           2) AS avg_weight,
                       ROUND(AVG(product_length_cm),          2) AS avg_length,
                       ROUND(AVG(product_height_cm),          2) AS avg_height,
                       ROUND(AVG(product_width_cm),           2) AS avg_width
                FROM base
                GROUP BY product_category_name
            )
            SELECT
                b.product_id,
                b.product_category_name,
                COALESCE(b.product_name_length,        CAST(ca.avg_name_len AS INT)) AS product_name_length,
                COALESCE(b.product_description_length, CAST(ca.avg_desc_len AS INT)) AS product_description_length,
                COALESCE(b.product_photos_qty,         CAST(ca.avg_photos   AS INT)) AS product_photos_qty,
                COALESCE(b.product_weight_g,           ca.avg_weight)                AS product_weight_g,
                COALESCE(b.product_length_cm,          ca.avg_length)                AS product_length_cm,
                COALESCE(b.product_height_cm,          ca.avg_height)                AS product_height_cm,
                COALESCE(b.product_width_cm,           ca.avg_width)                 AS product_width_cm,
                ROUND(
                    COALESCE(b.product_length_cm, ca.avg_length)
                    * COALESCE(b.product_height_cm, ca.avg_height)
                    * COALESCE(b.product_width_cm,  ca.avg_width), 2)                AS product_volume_cm3,
                CURRENT_TIMESTAMP()                                                  AS ingestion_timestamp
            FROM base b JOIN cat_avgs ca USING (product_category_name)
        """,
    },

    # Orders: mapping columns use derived aliases and prose that break dynamic SQL;
    # use full_sql to define all columns explicitly from the bronze source
    "sales.silver.orders": {
        "full_sql": """
            SELECT
                order_id,
                customer_id,
                order_status,
                CASE
                    WHEN LOWER(TRIM(order_status)) IN ('delivered','invoiced')   THEN 'completed'
                    WHEN LOWER(TRIM(order_status)) IN ('canceled','unavailable') THEN 'canceled'
                    ELSE 'pending'
                END AS order_status_group,
                order_purchase_timestamp,
                order_approved_at,
                order_delivered_carrier_date,
                order_delivered_customer_date,
                order_estimated_delivery_date,
                TO_DATE(order_purchase_timestamp)                              AS purchase_date,
                TO_DATE(order_approved_at)                                     AS approved_date,
                TO_DATE(order_delivered_carrier_date)                          AS carrier_date,
                TO_DATE(order_delivered_customer_date)                         AS delivered_date,
                TO_DATE(order_estimated_delivery_date)                         AS estimated_date,
                HOUR(order_purchase_timestamp)                                 AS purchase_hour,
                DATE_FORMAT(order_purchase_timestamp, 'E')                     AS purchase_weekday_name,
                CASE
                    WHEN HOUR(order_purchase_timestamp) BETWEEN 6  AND 11 THEN 'morning'
                    WHEN HOUR(order_purchase_timestamp) BETWEEN 12 AND 17 THEN 'afternoon'
                    WHEN HOUR(order_purchase_timestamp) BETWEEN 18 AND 22 THEN 'evening'
                    ELSE 'night'
                END AS purchase_part_of_day,
                CASE WHEN DAYOFWEEK(TO_DATE(order_purchase_timestamp)) IN (1,7)
                     THEN TRUE ELSE FALSE END                                  AS is_weekend_purchase,
                CASE WHEN order_delivered_customer_date IS NOT NULL
                     THEN HOUR(order_delivered_customer_date) END              AS delivered_hour,
                CASE WHEN order_delivered_customer_date IS NOT NULL
                     THEN DATE_FORMAT(order_delivered_customer_date, 'E') END  AS delivered_weekday_name,
                DATEDIFF(order_delivered_customer_date, order_purchase_timestamp)      AS delivery_days,
                DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) AS delay_days,
                (order_delivered_customer_date IS NULL)                        AS is_missing_customer_date,
                (order_delivered_carrier_date  IS NULL)                        AS is_missing_carrier_date,
                (order_approved_at             IS NULL)                        AS is_missing_approved,
                CASE WHEN order_approved_at < order_purchase_timestamp
                          OR order_delivered_carrier_date < order_approved_at
                          OR order_delivered_customer_date < order_delivered_carrier_date
                     THEN TRUE ELSE FALSE END                                  AS is_invalid_timeflow,
                CURRENT_TIMESTAMP()                                            AS ingestion_timestamp
            FROM sales.bronze.orders
        """,
    },

    # Order Items: remove duplicate (order_id, order_item_id) rows
    # col_overrides: correct malformed mapping expressions that can't be auto-fixed
    "sales.silver.order_items": {
        "dedup_partition": "order_id, order_item_id",
        "dedup_order_by":  "order_id, order_item_id",
        "col_overrides": {
            # Mapping says "YEAR(col) < 2017 OR > 2019" — missing left operand after OR
            "is_ship_date_outlier": "(YEAR(TO_TIMESTAMP(shipping_limit_date)) < 2017 OR YEAR(TO_TIMESTAMP(shipping_limit_date)) > 2019) AS `is_ship_date_outlier`",
        },
    },

    # Order Payments: transformation uses short aliases (installments, value) that
    # don't exist; override with the actual column names from the bronze table
    "sales.silver.order_payments": {
        "col_overrides": {
            "payment_per_installment": "(CASE WHEN payment_installments > 0 AND payment_value IS NOT NULL THEN ROUND(payment_value / payment_installments, 2) ELSE NULL END) AS `payment_per_installment`",
        },
    },

    # Order Reviews: keep only rows with valid scores
    # col_overrides: mapping uses semicolons and short aliases (title/message) — real column names required
    "sales.silver.order_reviews": {
        "where_clause": "review_score IS NOT NULL AND review_score BETWEEN 1 AND 5 AND order_id IS NOT NULL",
        "col_overrides": {
            "review_comment_title": "COALESCE(review_comment_title, CASE WHEN review_comment_message IS NOT NULL THEN SUBSTRING(review_comment_message, 1, 50) END) AS `review_comment_title`",
            "is_missing_comment":   "(review_comment_title IS NULL AND review_comment_message IS NULL) AS `is_missing_comment`",
            "has_title_only":       "(review_comment_title IS NOT NULL AND review_comment_message IS NULL) AS `has_title_only`",
            "has_message_only":     "(review_comment_title IS NULL AND review_comment_message IS NOT NULL) AS `has_message_only`",
        },
    },

    # Reviews bad rows: mapping has only a wildcard row — bypass the engine entirely
    "sales.silver.reviews_bad_rows": {
        "full_sql": """
            SELECT *,
                   CURRENT_TIMESTAMP() AS ingestion_timestamp
            FROM   sales.bronze.order_reviews
            WHERE  review_score IS NULL
               OR  TRY_CAST(review_score AS INT) NOT BETWEEN 1 AND 5
               OR  order_id IS NULL
        """,
    },

    # Geolocation: centroid aggregation per zip-code prefix
    # FIRST() is required for city/state — they are non-aggregate cols in a GROUP BY
    "sales.silver.geolocation": {
        "full_sql": """
            SELECT
                CAST(geolocation_zip_code_prefix AS INT)                                    AS geolocation_zip_code_prefix,
                ROUND(AVG(CAST(geolocation_lat AS DOUBLE)), 6)                              AS geolocation_lat,
                ROUND(AVG(CAST(geolocation_lng AS DOUBLE)), 6)                              AS geolocation_lng,
                FIRST(INITCAP(LOWER(TRIM(REGEXP_REPLACE(geolocation_city,'[^a-zA-Z\\s]','')))))
                                                                                            AS geolocation_city,
                FIRST(UPPER(TRIM(geolocation_state)))                                       AS geolocation_state,
                CURRENT_TIMESTAMP()                                                         AS ingestion_timestamp
            FROM sales.bronze.geolocation
            GROUP BY geolocation_zip_code_prefix
        """,
    },
}


def apply_bronze_to_silver(mapping_df, overrides):
    """
    Iterate over every Silver target table in the mapping and write a Delta table.
    Column expressions are generated dynamically from the Excel mapping rows;
    table-level overrides inject complex SQL where needed.
    """
    totals = {}

    for target_table, grp in mapping_df.groupby("target_table"):
        target_table = str(target_table).strip()
        ov = overrides.get(target_table, {})
        print(f"\n── {target_table}")

        # A: full SQL override (products, geolocation)
        if "full_sql" in ov:
            df = spark.sql(ov["full_sql"].strip())
            df.write.format("delta").mode("overwrite").saveAsTable(target_table)
            totals[target_table] = df.count()
            print(f"   ✓ full_sql → {totals[target_table]:,} rows")
            continue

        # B: build SELECT expressions from mapping rows
        source_table   = str(grp["source_table"].dropna().iloc[0]).strip()
        col_ovr        = ov.get("col_overrides", {})
        exprs          = []
        for _, row in grp.iterrows():
            tgt_name = str(row.get("target_col", "") or "").strip()
            if tgt_name in col_ovr:
                exprs.append(col_ovr[tgt_name])   # use explicit override
            else:
                e = to_sql_expr(row)
                if e:
                    exprs.append(e)

        # Inject extra derived columns (orders: status_group, time_of_day, is_weekend)
        for extra in ov.get("extra_cols", []):
            exprs.append(extra.strip())

        # Ensure audit timestamp
        if not any("ingestion_timestamp" in e for e in exprs):
            exprs.append("CURRENT_TIMESTAMP() AS `ingestion_timestamp`")

        sel   = ",\n    ".join(exprs)
        where = ov.get("where_clause", "")

        # C: ROW_NUMBER deduplication (sellers, order_items)
        if "dedup_partition" in ov:
            part = ov["dedup_partition"]
            ob   = ov.get("dedup_order_by", part)
            sql  = f"""
                WITH ranked AS (
                    SELECT {sel},
                           ROW_NUMBER() OVER (PARTITION BY {part} ORDER BY {ob}) AS _rn
                    FROM {source_table}
                )
                SELECT * FROM ranked WHERE _rn = 1
            """
        else:
            where_frag = f"\nWHERE {where}" if where else ""
            sql = f"SELECT DISTINCT\n    {sel}\nFROM {source_table}{where_frag}"

        df = spark.sql(sql.strip())

        df.write.format("delta").mode("overwrite").saveAsTable(target_table)
        totals[target_table] = df.count()
        print(f"   ✓ {totals[target_table]:,} rows")

    return totals


bts_totals = apply_bronze_to_silver(bts, BTS_OVERRIDES)
print(f"\n✅  Bronze → Silver complete — {len(bts_totals)} tables written")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Gold tables use full_sql overrides because they involve multi-table JOINs,
# CTEs, and window functions over Silver.  The mapping drives WHAT to compute;
# the SQL below is the HOW.
# ══════════════════════════════════════════════════════════════════════════════

STG_OVERRIDES = {

    "sales.gold.customer_top_5_states_by_city_diversity": {"full_sql": """
        SELECT customer_state,
               COUNT(DISTINCT customer_unique_id) AS customer_count,
               COUNT(DISTINCT customer_city)      AS distinct_cities,
               CURRENT_TIMESTAMP()               AS ingestion_timestamp
        FROM   sales.silver.customers
        GROUP BY customer_state
        ORDER BY distinct_cities DESC
        LIMIT 5
    """},

    "sales.gold.customer_distribution_by_state": {"full_sql": """
        WITH agg AS (
            SELECT customer_state,
                   COUNT(DISTINCT customer_unique_id) AS customer_count,
                   COUNT(DISTINCT customer_city)      AS distinct_cities
            FROM   sales.silver.customers
            GROUP BY customer_state
        )
        SELECT customer_state,
               customer_count,
               distinct_cities,
               ROW_NUMBER() OVER (ORDER BY customer_count DESC) AS state_rank,
               CURRENT_TIMESTAMP() AS ingestion_timestamp
        FROM agg
    """},

    "sales.gold.seller_city_stats": {"full_sql": """
        SELECT seller_state,
               seller_city,
               COUNT(*) AS seller_count,
               CURRENT_TIMESTAMP()       AS ingestion_timestamp
        FROM   sales.silver.sellers
        GROUP BY seller_state, seller_city
        ORDER BY seller_state, seller_count DESC
    """},

    "sales.gold.product_category_summary": {"full_sql": """
        SELECT product_category_name,
               COUNT(product_id)                                              AS product_count,
               ROUND(AVG(product_weight_g),   2)                             AS avg_weight_g,
               ROUND(AVG(product_length_cm),  2)                             AS avg_length_cm,
               ROUND(AVG(product_height_cm),  2)                             AS avg_height_cm,
               ROUND(AVG(product_width_cm),   2)                             AS avg_width_cm,
               ROUND(AVG(product_volume_cm3), 2)                             AS avg_volume_cm3,
               ROUND(AVG(product_weight_g / NULLIF(product_volume_cm3,0)),3) AS avg_density,
               CURRENT_TIMESTAMP()                                           AS ingestion_timestamp
        FROM   sales.silver.products
        WHERE  product_category_name IS NOT NULL
        GROUP BY product_category_name
    """},

    # NOTE: runs after product_category_summary (sorted table order ensures this)
    "sales.gold.product_category_summary_en": {"full_sql": """
        SELECT pcs.product_category_name                                           AS category_pt,
               COALESCE(cl.category_en, pcs.product_category_name)                AS category_en,
               INITCAP(REGEXP_REPLACE(
                   COALESCE(cl.category_en, pcs.product_category_name),'_',' '))   AS category_en_display,
               pcs.product_count,
               pcs.avg_weight_g,
               pcs.avg_length_cm,
               pcs.avg_height_cm,
               pcs.avg_width_cm,
               pcs.avg_volume_cm3,
               pcs.avg_density,
               CURRENT_TIMESTAMP()                                                 AS ingestion_timestamp
        FROM   sales.gold.product_category_summary  pcs
        LEFT JOIN sales.silver.category_lookup       cl
               ON pcs.product_category_name = cl.category_pt
    """},

    "sales.gold.order_performance": {"full_sql": """
        WITH item_agg AS (
            SELECT order_id,
                   ROUND(SUM(price + freight_value), 2) AS order_value,
                   COUNT(*)                             AS item_count
            FROM   sales.silver.order_items
            GROUP BY order_id
        ),
        pay_agg AS (
            SELECT order_id,
                   FIRST(payment_type)                          AS payment_type,
                   ROUND(SUM(payment_value), 2)                 AS total_payment,
                   ROUND(AVG(NULLIF(payment_installments,0)),2) AS avg_installments
            FROM   sales.silver.order_payments
            GROUP BY order_id
        ),
        rev_agg AS (
            SELECT order_id,
                   ROUND(AVG(review_score),2) AS avg_review_score
            FROM   sales.silver.order_reviews
            GROUP BY order_id
        )
        SELECT o.order_id,
               TO_DATE(o.order_purchase_timestamp)  AS purchase_date,
               o.order_status_group,
               o.delivery_days,
               o.delay_days,
               ia.order_value,
               ia.item_count,
               pa.payment_type,
               pa.total_payment,
               pa.avg_installments,
               ra.avg_review_score,
               CURRENT_TIMESTAMP()                  AS ingestion_timestamp
        FROM   sales.silver.orders          o
        LEFT JOIN item_agg                  ia USING (order_id)
        LEFT JOIN pay_agg                   pa USING (order_id)
        LEFT JOIN rev_agg                   ra USING (order_id)
    """},
}


def apply_silver_to_gold(mapping_df, overrides):
    """
    For each Gold table: use full_sql override when present;
    otherwise build a dynamic GROUP BY aggregation from the mapping columns.
    Tables are processed in sorted order so dependencies resolve correctly
    (e.g. product_category_summary before product_category_summary_en).
    """
    totals = {}
    all_tables = sorted(mapping_df["target_table"].dropna().unique())

    for target_table in all_tables:
        target_table = str(target_table).strip()
        ov  = overrides.get(target_table, {})
        grp = mapping_df[mapping_df["target_table"].str.strip() == target_table]
        print(f"\n── {target_table}")

        # A: full SQL override
        if "full_sql" in ov:
            df = spark.sql(ov["full_sql"].strip())
            df.write.format("delta").mode("overwrite").saveAsTable(target_table)
            totals[target_table] = df.count()
            print(f"   ✓ full_sql → {totals[target_table]:,} rows")
            continue

        # B: dynamic aggregation fallback (single-source tables only)
        source_tables = grp["source_tables"].dropna().unique()
        if len(source_tables) != 1:
            print(f"   ⚠  multi-source — add to STG_OVERRIDES and rerun")
            continue

        source_tbl = str(source_tables[0]).strip()
        grain_cols = []
        for g in grp["grain"].dropna():
            for c in str(g).split(","):
                c = c.strip()
                if c and c not in grain_cols:
                    grain_cols.append(c)

        exprs = [f"`{c}`" for c in grain_cols]
        for _, row in grp.iterrows():
            lgc = str(row.get("transformation", "") or "").strip()
            tgt = str(row.get("target_col",     "") or "").strip()
            if not tgt or not lgc or tgt in grain_cols:
                continue
            if any(fn in lgc.upper() for fn in ["COUNT(","AVG(","SUM(","MAX(","MIN(","ROUND("]):
                exprs.append(f"({lgc}) AS `{tgt}`")

        exprs.append("CURRENT_TIMESTAMP() AS `ingestion_timestamp`")
        grp_clause = ", ".join(f"`{c}`" for c in grain_cols)
        sel        = ",\n    ".join(exprs)
        sql        = f"SELECT {sel}\nFROM {source_tbl}" + (f"\nGROUP BY {grp_clause}" if grp_clause else "")

        df = spark.sql(sql.strip())
        df.write.format("delta").mode("overwrite").saveAsTable(target_table)
        totals[target_table] = df.count()
        print(f"   ✓ {totals[target_table]:,} rows")

    return totals


stg_totals = apply_silver_to_gold(stg, STG_OVERRIDES)
print(f"\n✅  Silver → Gold complete — {len(stg_totals)} tables written")